# 01 — Data Collection

This notebook guides you through capturing labeled hand gesture images directly from your webcam.

**Pipeline step:** Webcam → Captured Images stored under `data/<class_index>/`

> **Tip:** You can also run the standalone script from the terminal:
> ```bash
> python src/collect_data.py --classes 3 --size 150
> ```

## 1.1 Configuration

All constants are imported from `src/config.py`. Edit that file to change classes, dataset size, or the label names.

In [ ]:
import os
import sys

# Make the project root importable
sys.path.insert(0, os.path.abspath('..'))

import cv2
from src.config import CAMERA_INDEX, DATA_DIR, DATASET_SIZE, LABELS_DICT, NUM_CLASSES

print(f'Data directory : {DATA_DIR}')
print(f'Classes        : {NUM_CLASSES}')
print(f'Images / class : {DATASET_SIZE}')
print(f'Label mapping  : {LABELS_DICT}')

## 1.2 Create Output Directories

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
for i in range(NUM_CLASSES):
    class_dir = os.path.join(DATA_DIR, str(i))
    os.makedirs(class_dir, exist_ok=True)
    print(f'  Created: {class_dir}')

## 1.3 Capture Images

For **each class** the script will:
1. Open a preview window — position your hand, then **press `Q`** to start capturing.
2. Automatically save `DATASET_SIZE` frames to `data/<class_index>/`.

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)

for class_idx in range(NUM_CLASSES):
    class_dir = os.path.join(DATA_DIR, str(class_idx))
    label_name = LABELS_DICT.get(class_idx, str(class_idx))
    print(f'\nCollecting data for class {class_idx}: "{label_name}"')
    print('  Get into position, then press Q to start capturing ...')

    # Wait for user readiness
    while True:
        ret, frame = cap.read()
        cv2.putText(frame, f'Class: {label_name}  |  Press Q to start',
                    (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 220, 0), 2, cv2.LINE_AA)
        cv2.imshow('Data Collection', frame)
        if cv2.waitKey(20) == ord('q'):
            break

    # Capture frames
    counter = 0
    while counter < DATASET_SIZE:
        ret, frame = cap.read()
        cv2.putText(frame, f'Capturing: {counter + 1}/{DATASET_SIZE}',
                    (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 180, 255), 2, cv2.LINE_AA)
        cv2.imshow('Data Collection', frame)
        cv2.waitKey(20)
        cv2.imwrite(os.path.join(class_dir, f'{counter}.jpg'), frame)
        counter += 1

    print(f'  Saved {counter} images to {class_dir}')

cap.release()
cv2.destroyAllWindows()
print('\nData collection complete!')

## 1.4 Verify Dataset

Quickly confirm how many images were captured per class.

In [ ]:
print('Dataset summary:')
total = 0
for class_idx in range(NUM_CLASSES):
    class_dir = os.path.join(DATA_DIR, str(class_idx))
    count = len([f for f in os.listdir(class_dir) if f.endswith('.jpg')])
    total += count
    print(f'  Class {class_idx} ({LABELS_DICT.get(class_idx)}): {count} images')
print(f'  Total: {total} images')